In [1]:
import os
import random
import torch
import nltk
from nltk.corpus import stopwords

import numpy as np
import pandas as pd

from transformers import AutoModelForSequenceClassification, AutoTokenizer
from peft import PeftModel

from interpreto import (
    IntegratedGradients,
    KernelShap,
    Lime,
    Occlusion,
    Saliency,
    SmoothGrad,
    Sobol,
    plot_attributions,
)
from interpreto.attributions.metrics import Deletion, Insertion

from collections import defaultdict

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [2]:
# Download stopwords.
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to /home/isabel/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:
# Set up constant variables.
INPUT_FOLDER = 'data'
OUTPUT_FOLDER = 'output_explanations'
DEVICE = 0 if torch.cuda.is_available() else -1

# Make model variables.
MODEL_NAME = "nlpie/tiny-clinicalbert"
PEFT_HEAD = "synth_lora_model_tinyclinicalbert"


LABEL_MAP = {
    0: "met",
    1: "unmet"
}

In [4]:
# Make output dir.
os.makedirs(f'./{OUTPUT_FOLDER}', exist_ok=True)

In [5]:
# Set random states.
def set_random_states(random_state):
    # Set various random seeds.
    np.random.seed(random_state)
    random.seed(random_state)
    torch.manual_seed(random_state)
    torch.cuda.manual_seed_all(random_state)
    os.environ["PYTHONHASHSEED"] = str(random_state)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    return random_state
RANDOM_STATE = set_random_states(1618)

In [ ]:
# Get real data to work with.
all_data = {}
for folder in os.listdir(f'../../../{INPUT_FOLDER}/'):
    if '.' not in folder:
        for sub_folder in os.listdir(f'../../../{INPUT_FOLDER}/{folder}'):
            if '.' not in sub_folder and 'T1' in sub_folder:
                for sub_sub_folder in os.listdir(f'../../../{INPUT_FOLDER}/{folder}/{sub_folder}'):
                    if '.' not in sub_sub_folder: 
                        daily_nurse = pd.read_excel(f'../../../{INPUT_FOLDER}/{folder}/{sub_folder}/{sub_sub_folder}/dailyNurseNotes_{sub_sub_folder.split(' ')[0]}.xlsx')
                        daily_nurse = daily_nurse.dropna(subset=['Note', 'Date', 'Time'])
                        all_data[f"{sub_sub_folder.split(' ')[0]}"] = daily_nurse['Note'].dropna().values.tolist()

In [7]:
# Prepare dataset.
inputs = []

for patient_id in all_data:
    inputs.extend(all_data[patient_id])

print(f"Number of notes: {len(inputs)}")

Number of notes: 12373


In [8]:
# Load model.
base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

model_classif = PeftModel.from_pretrained(base_model, PEFT_HEAD)
tokenizer_classif = AutoTokenizer.from_pretrained(PEFT_HEAD)

dico_name_classes = LABEL_MAP

Loading weights:   0%|          | 0/69 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: nlpie/tiny-clinicalbert
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.weight                   | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expe

In [9]:
# Initialize explainers.
methods_list = [IntegratedGradients, KernelShap, Lime, Occlusion, Saliency, SmoothGrad, Sobol]
explainers = [
    method(model_classif, tokenizer_classif)
    for method in methods_list
]
# Initialize deletion metric (use the same model as for the explainers) - https://for-sight-ai.github.io/interpreto/api/attributions/metrics/deletion/
deletion_metric = Deletion(model_classif, tokenizer_classif)
# Initialize insertion metric (use the same model as for the explainers) - https://for-sight-ai.github.io/interpreto/api/attributions/metrics/insertion/
insertion_metric = Insertion(model_classif, tokenizer_classif)

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


In [10]:
explainer_results = {}
for explainer in explainers:
    # Get explainer name.
    explainer_name = str(explainer).replace('<interpreto.attributions.base.Classification', '').split()[0]
    print(f" -------- {explainer_name} -------- ")
    # Inputs can also be a list of strings, or even `input_ids`
    attributions = explainer(inputs)
    # compute scores on attributions
    deletion_auc, deletion_detailed_scores = deletion_metric.evaluate(attributions)
    print(f"{explainer_name} Deletion AUC: {round(deletion_auc, 3)} (lower is better - bounded between 0 and 1)")
    insertion_auc, insertion_detailed_scores = insertion_metric.evaluate(attributions)
    print(f"{explainer_name} Insertion AUC: {round(insertion_auc, 3)} (higher is better - bounded between 0 and 1))")
    explainer_results[explainer_name] = {'Attributions': attributions, 'Deletion AUC': deletion_auc, 'Deletion Detailed Scores': deletion_detailed_scores, 'Insertion AUC': insertion_auc, 'Insertion Detailed Scores': insertion_detailed_scores}

 -------- IntegratedGradients -------- 
IntegratedGradients Deletion AUC: 0.877 (lower is better - bounded between 0 and 1)
IntegratedGradients Insertion AUC: 0.94 (higher is better - bounded between 0 and 1))
 -------- KernelShap -------- 
KernelShap Deletion AUC: 0.616 (lower is better - bounded between 0 and 1)
KernelShap Insertion AUC: 0.979 (higher is better - bounded between 0 and 1))
 -------- Lime -------- 
Lime Deletion AUC: 0.677 (lower is better - bounded between 0 and 1)
Lime Insertion AUC: 0.977 (higher is better - bounded between 0 and 1))
 -------- Occlusion -------- 
Occlusion Deletion AUC: 0.672 (lower is better - bounded between 0 and 1)
Occlusion Insertion AUC: 0.977 (higher is better - bounded between 0 and 1))
 -------- Saliency -------- 
Saliency Deletion AUC: 0.886 (lower is better - bounded between 0 and 1)
Saliency Insertion AUC: 0.941 (higher is better - bounded between 0 and 1))
 -------- SmoothGrad -------- 
SmoothGrad Deletion AUC: 0.889 (lower is better - 

In [11]:
# Extract tokens and scores.
results_dict = explainer_results
top_terms = {}

for method, data in results_dict.items():

    class_scores = {
        0: defaultdict(list),
        1: defaultdict(list)
    }

    for attr_output in data["Attributions"]:

        tokens = attr_output.elements
        scores = attr_output.attributions.squeeze().cpu().numpy()
        label = int(attr_output.targets.item())

        for tok, score in zip(tokens, scores):

            tok = tok.lower()

            if tok.isalpha():   # remove punctuation
                class_scores[label][tok].append(score)

    # compute averages
    class_avg = {}
    for c in [0,1]:
        class_avg[c] = {
            tok: np.mean(vals)
            for tok, vals in class_scores[c].items()
        }

    top_terms[method] = class_avg

In [12]:
# Get top tokens per class.
def get_top_tokens(token_dict, k=10, remove_stopwords=True):
    new_token_dict = {}
    if remove_stopwords:
        for key, value in token_dict.items():
            if key in stop_words:
                pass
            else:
                new_token_dict[key] = value
    else:
        new_token_dict = token_dict
    return sorted(
        new_token_dict.items(),
        key=lambda x: abs(x[1]),
        reverse=True
    )[:k]

# Make clean table.
rows = []

for method in top_terms:

    for c in [0,1]:

        for tok, score in get_top_tokens(top_terms[method][c], 15):

            rows.append({
                "method": method,
                "class": c,
                "token": tok,
                "score": score
            })

df = pd.DataFrame(rows)
df.to_csv(f'./{OUTPUT_FOLDER}/attribution_per_class_per_algorithm_results.csv')

In [14]:
# Get results from individual samples.
def get_top_tokens_from_attributions(attribution_outputs, top_k=10, remove_stopwords=True):
    top_tokens_per_sample = []

    for output in attribution_outputs:
        attributions = output.attributions.squeeze(0)  # shape: [num_tokens]
        tokens = output.elements  # list of strings

        # Remove stopwords.
        if remove_stopwords:
            filtered = [(tok, attr.item()) for tok, attr in zip(tokens, attributions) if tok.lower() not in stop_words]
        else:
            filtered = [(tok, attr.item()) for tok, attr in zip(tokens, attributions)]

        # Sort by absolute attribution values.
        filtered.sort(key=lambda x: abs(x[1]), reverse=True)

        # Take top_k.
        top_tokens = dict(filtered[:top_k])
        top_tokens_per_sample.append(top_tokens)

    return top_tokens_per_sample


# Loop over all attribution methods.
rows = []
for method in results_dict.keys():
    attribution_outputs = results_dict[method]['Attributions']
    
    # Get top tokens per sample.
    top_tokens_per_sample = get_top_tokens_from_attributions(
        attribution_outputs, top_k=5, remove_stopwords=True
    )
    
    # Loop through samples and tokens.
    for sample_idx, (token_dict, output) in enumerate(zip(top_tokens_per_sample, attribution_outputs)):
        # Reconstruct sentence to save in file.
        sentence = " ".join(output.elements)
        # Get output class.
        predicted_class = output.classes.item()

        for token, score in token_dict.items():
            rows.append({
                "method": method,
                'sentence':sentence,
                'predicted_class': predicted_class,
                "sample": sample_idx,
                "token": token,
                "score": score
            })

# Convert to DataFrame.
df_top_tokens = pd.DataFrame(rows)
df_top_tokens.to_csv(f"./{OUTPUT_FOLDER}/attribution_per_sample_per_algorithm_results.csv", index=False)
